# Homework 2 — Vector Search
**LLM Zoomcamp 2026 | Cohort 2026**

Stack: ONNX embeddings · minsearch · numpy · RRF hybrid search

## Setup: cargamos el Embedder

El `Embedder` es nuestra herramienta para convertir texto → vector de 384 números.
Lee dos archivos que ya descargamos:
- `tokenizer.json` → convierte texto en tokens (números enteros)
- `model.onnx`     → convierte tokens en el vector final de 384 dimensiones

In [1]:
# Importamos la clase Embedder que está en el archivo embedder.py
# (lo copiamos al mismo directorio que este notebook)
from embedder import Embedder

# Creamos una instancia del embedder
# Por defecto busca el modelo en: models/Xenova/all-MiniLM-L6-v2/
embedder = Embedder()

print("Embedder listo!")

Embedder listo!


## Q1. Embedding de un query

**Pregunta:** Embeddea el siguiente query y dime cuál es el primer valor del vector (`v[0]`).

> *How does approximate nearest neighbor search work?*

Opciones: -0.31 / -0.02 / 0.12 / 0.44

In [2]:
# El query que vamos a convertir en vector
query = "How does approximate nearest neighbor search work?"

# encode() recibe un string y devuelve un array de numpy con 384 números
# normalize=True (por defecto) → el vector tiene longitud 1
# Eso nos permite usar .dot() directamente como cosine similarity
v = embedder.encode(query)

# Verificamos que sea un vector de 384 dimensiones
print(f"Forma del vector: {v.shape}")    # debe decir (384,)
print(f"Primer valor v[0]: {v[0]:.4f}")  # ← respuesta a Q1
print(f"Norma del vector: {v.dot(v):.4f}") # debe ser ~1.0 (está normalizado)

Forma del vector: (384,)
Primer valor v[0]: -0.0206
Norma del vector: 1.0000


In [3]:
# Cargamos los documentos del curso desde GitHub
# gitsource descarga solo los archivos .md que estén en /lessons/
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",        # versión fija → todos trabajamos con los mismos datos
    allowed_extensions={"md"},  # solo archivos markdown
    filename_filter=lambda path: "/lessons/" in path,  # solo carpetas lessons/
)

# Leemos todos los archivos y los convertimos a diccionarios {filename, content}
documents = [file.parse() for file in reader.read()]

print(f"Total de páginas cargadas: {len(documents)}")
print(f"Ejemplo de documento: {list(documents[0].keys())}")


Total de páginas cargadas: 72
Ejemplo de documento: ['content', 'filename']


In [4]:
# Buscamos la página específica que pide el homework
# next() recorre la lista y para cuando encuentra el documento con ese filename
pagina = next(
    doc for doc in documents
    if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)

# Embeddeamos el contenido completo de esa página
# encode() → 1 vector de 384 números, normalizado
v_pagina = embedder.encode(pagina["content"])

# Cosine similarity = producto punto entre dos vectores normalizados
# v       → vector del query
# v_pagina → vector de la página
similitud = v.dot(v_pagina)

print(f"Cosine similarity: {similitud:.4f}")


Cosine similarity: 0.3611


In [5]:
from gitsource import chunk_documents

# Partimos cada página en chunks de 2000 chars, solapando 1000
# size=2000  → cada chunk tiene máximo 2000 caracteres
# step=1000  → la ventana avanza 1000 chars entre chunks
chunks = chunk_documents(documents, size=2000, step=1000)

print(f"Total de chunks: {len(chunks)}")
print(f"\nEjemplo de chunk:")
print(f"  filename : {chunks[0]['filename']}")
print(f"  start    : {chunks[0]['start']}")   # en qué posición de la página empieza
print(f"  content  : {chunks[0]['content'][:80]}...")  # primeros 80 chars


Total de chunks: 295

Ejemplo de chunk:
  filename : 01-agentic-rag/lessons/01-intro.md
  start    : 0
  content  : # Introduction

Video: [Watch this lesson](https://www.youtube.com/watch?v=rQYyF...


In [6]:
import numpy as np

# encode_batch es más eficiente que llamar encode() 295 veces
# Procesa todos los chunks de una sola vez en el modelo ONNX
# Resultado: matriz numpy de forma (295, 384)
print("Embeddeando 295 chunks...")
X = embedder.encode_batch([chunk["content"] for chunk in chunks])

print(f"Forma de la matriz X: {X.shape}")  # debe ser (295, 384)

# Calculamos la similitud del query contra TODOS los chunks a la vez
# X.dot(v) → multiplica cada fila de X por v → 295 scores de una vez
scores = X.dot(v)

print(f"Forma de scores: {scores.shape}")  # debe ser (295,)

# np.argmax → índice del score más alto
mejor_idx = np.argmax(scores)

print(f"\nMejor score: {scores[mejor_idx]:.4f}")
print(f"Chunk más relevante: {chunks[mejor_idx]['filename']}")


Embeddeando 295 chunks...
Forma de la matriz X: (295, 384)
Forma de scores: (295,)

Mejor score: 0.6489
Chunk más relevante: 02-vector-search/lessons/07-sqlitesearch-vector.md


In [8]:
from minsearch import VectorSearch

# VectorSearch solo necesita saber qué campos son keywords (metadata)
# Los vectores se pasan por separado en .fit()
index = VectorSearch(keyword_fields=["filename"])

# fit() recibe DOS argumentos separados:
#   X      → la matriz (295, 384) con todos los vectores — ya la tenemos calculada
#   chunks → la lista de dicts con filename, content, start (el "payload")
index.fit(X, chunks)
print("Índice listo")

# Ahora buscamos con el nuevo query que pide el HW2
query_q4 = "What metric do we use to evaluate a search engine?"

# Embeddeamos el nuevo query
v_q4 = embedder.encode(query_q4)

# search() recibe el vector del query y devuelve los resultados ordenados por score
results = index.search(v_q4, num_results=5)

# El primer resultado es la respuesta a Q4
print(f"\nPrimer resultado: {results[0]['filename']}")

# Mostramos los top 5 para tener contexto
print("\nTop 5:")
for i, r in enumerate(results):
    print(f"  {i+1}. {r['filename']}")


Índice listo

Primer resultado: 04-evaluation/lessons/05-search-metrics.md

Top 5:
  1. 04-evaluation/lessons/05-search-metrics.md
  2. 04-evaluation/lessons/01-intro.md
  3. 01-agentic-rag/lessons/05-search.md
  4. 04-evaluation/lessons/01-intro.md
  5. 04-evaluation/lessons/15-next-steps.md


In [9]:
from minsearch import Index

# Creamos el índice de keyword search (TF-IDF clásico)
# text_fields → campos donde se busca por palabras
# keyword_fields → metadata que se devuelve pero no se busca
keyword_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

# Indexamos los mismos 295 chunks
keyword_index.fit(chunks)
print("Índice keyword listo")

query_q5 = "How do I store vectors in PostgreSQL?"

# Búsqueda por keywords (palabras exactas, TF-IDF)
keyword_results = keyword_index.search(query_q5, num_results=5)

# Búsqueda vectorial (significado semántico) — v_q4 no sirve, es otro query
v_q5 = embedder.encode(query_q5)
vector_results = index.search(v_q5, num_results=5)

# Comparamos los top 5 de cada método
print("\nKeyword search top 5:")
keyword_files = [r["filename"] for r in keyword_results]
for f in keyword_files:
    print(f"  {f}")

print("\nVector search top 5:")
vector_files = [r["filename"] for r in vector_results]
for f in vector_files:
    print(f"  {f}")

print("\nEn vector pero NO en keyword:")
solo_vector = [f for f in vector_files if f not in keyword_files]
for f in solo_vector:
    print(f"  ★ {f}")


Índice keyword listo

Keyword search top 5:
  02-vector-search/lessons/02-embeddings.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/01-intro.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/01-intro.md

Vector search top 5:
  02-vector-search/lessons/08-pgvector.md
  02-vector-search/lessons/08-pgvector.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/08-pgvector.md
  02-vector-search/lessons/08-pgvector.md

En vector pero NO en keyword:
  ★ 02-vector-search/lessons/08-pgvector.md
  ★ 02-vector-search/lessons/08-pgvector.md
  ★ 02-vector-search/lessons/08-pgvector.md
  ★ 02-vector-search/lessons/08-pgvector.md


In [10]:
# Función RRF exactamente como la define el homework
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    # Recorremos cada lista de resultados (keyword y vector)
    for results in result_lists:
        # Recorremos cada documento con su posición (rank)
        for rank, doc in enumerate(results):
            # Clave única: (filename, start) para identificar cada chunk
            key = (doc["filename"], doc["start"])
            # Sumamos 1/(k + rank) → posición 0 aporta más que posición 4
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    # Ordenamos por score descendente y devolvemos los top num_results
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


query_q6 = "How do I give the model access to tools?"

# Buscamos con los dos métodos
v_q6 = embedder.encode(query_q6)

vector_q6  = index.search(v_q6, num_results=5)
keyword_q6 = keyword_index.search(query_q6, num_results=5)

# Fusionamos con RRF
results_rrf = rrf([vector_q6, keyword_q6])

print("Resultado Q6 — primero tras RRF:")
print(f"  ★ {results_rrf[0]['filename']}")

print("\nTop 5 fusionados:")
for i, r in enumerate(results_rrf):
    print(f"  {i+1}. {r['filename']}")


Resultado Q6 — primero tras RRF:
  ★ 01-agentic-rag/lessons/13-function-calling.md

Top 5 fusionados:
  1. 01-agentic-rag/lessons/13-function-calling.md
  2. 01-agentic-rag/lessons/01-intro.md
  3. 01-agentic-rag/lessons/14-agentic-loop.md
  4. 04-evaluation/lessons/02-ground-truth.md
  5. 01-agentic-rag/lessons/16-other-frameworks.md
